<a href="https://colab.research.google.com/github/rzangef21/spb-kelompok-2/blob/main/Certainty_Factor/Certainty_Factor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# 1. Load dan Preprocessing Dasar
try:
    df = pd.read_csv('Dataset_indomaret_sales.csv')
    # Membersihkan data Units_Sold yang N/A seperti di screenshot sebelumnya
    df['Units_Sold'] = pd.to_numeric(df['Units_Sold'], errors='coerce')
    df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())
    df['Date'] = pd.to_datetime(df['Date'])
    df['Day'] = df['Date'].dt.day
    print("Dataset berhasil dimuat dan dibersihkan.\n")
except Exception as e:
    print(f"Error: {e}")
    exit()

# 2. Fungsi Logika Certainty Factor
def hitung_cf_jawaban(evidence_status, mb_pakar, md_pakar=0.1):
    """
    evidence_status: True jika data mendukung, False jika tidak
    mb_pakar: Measure of Belief (seberapa yakin pakar jika data benar)
    md_pakar: Measure of Disbelief (faktor ketidakyakinan)
    """
    if evidence_status:
        cf = (mb_pakar - md_pakar)
    else:
        # Jika fakta lapangan salah, nilai kepastian menjadi negatif
        cf = (0 - mb_pakar)
    return round(cf, 3)

# --- ANALISIS DATA UNTUK EVIDENCE ---

# Fakta 1: Tren Penjualan Harian
daily_sales = df.groupby('Date')['Total_Revenue'].sum()
# Cek apakah hari terakhir lebih tinggi dari rata-rata hari sebelumnya
tren_naik = daily_sales.iloc[-1] > daily_sales.mean()

# Fakta 2: Penjualan Tanggal 3
sales_per_day = df.groupby('Day')['Total_Revenue'].sum()
is_tgl_3_max = sales_per_day.idxmax() == 3

# Fakta 3: Jakarta Terendah
store_perf = df.groupby('Store_Location')['Total_Revenue'].sum()
is_jakarta_min = store_perf.idxmin() == 'Jakarta'

# --- OUTPUT DENGAN METODE CF ---

print("=== HASIL ANALISIS CERTAINTY FACTOR ===\n")

# Jawaban 1
cf_1 = hitung_cf_jawaban(tren_naik, 0.8)
status_1 = "Naik" if cf_1 > 0 else "Tidak Naik/Turun"
print(f"1. Apakah tren harian naik? \n   Status: {status_1} (CF: {cf_1})")

# Jawaban 2
cf_2 = hitung_cf_jawaban(is_tgl_3_max, 0.9)
status_2 = "Ya" if cf_2 > 0 else "Tidak"
print(f"2. Apakah tanggal 3 tertinggi? \n   Status: {status_2} (CF: {cf_2})")

# Jawaban 3
cf_3 = hitung_cf_jawaban(is_jakarta_min, 0.85)
status_3 = "Ya" if cf_3 > 0 else "Tidak"
print(f"3. Apakah Jakarta terendah? \n   Status: {status_3} (CF: {cf_3})")

# Penjelasan Tambahan untuk Bukti
print("\n--- Bukti Data ---")
print(f"- Penjualan Tertinggi ada pada Tanggal: {sales_per_day.idxmax()}")
print(f"- Lokasi Penjualan Terendah: {store_perf.idxmin()}")

Dataset berhasil dimuat dan dibersihkan.

=== HASIL ANALISIS CERTAINTY FACTOR ===

1. Apakah tren harian naik? 
   Status: Naik (CF: 0.7)
2. Apakah tanggal 3 tertinggi? 
   Status: Tidak (CF: -0.9)
3. Apakah Jakarta terendah? 
   Status: Tidak (CF: -0.85)

--- Bukti Data ---
- Penjualan Tertinggi ada pada Tanggal: 6
- Lokasi Penjualan Terendah: Bandung
